# 01 · Copernicus Browser: calibrated Sentinel-2 NDVI

**Objective:** Acquire B04/B08, crop a small shared AOI, calibrate surface reflectance, mask poor observations, and calculate NDVI.

**Route:** Manual acquisition / local files · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** an AOI NDVI map with calibration and mask information. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/cdse-sentinel-hub.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Obtain matching files

Open [Copernicus Browser](https://browser.dataspace.copernicus.eu/), sign in with CDSE, choose Sentinel-2 L2A, select your AOI and date, inspect clouds, and download the original product/bands plus metadata. Preserve the download product ID; a color visualization export is not a reflectance band.

Save B04 and B08 at the **same 10 m grid** from one L2A product; retain its product-level `MTD_MSIL2A.xml`, SCL raster, acquisition timestamp and product identifier. Set `INPUT_B04`, `INPUT_B08`, `INPUT_L2A_METADATA`, and `INPUT_SCL` in `.env`. Set `S2_AOI_BBOX` to a small rectangle **inside your own downloaded tile**; the southern Finland default is illustrative and may not intersect it. The original example's files are not redistributed.

Choose `S2_CALIBRATION=native_dn` for SAFE band digital numbers. Choose `harmonized_dn` only when the provider explicitly documents removed additive offsets and supplies `S2_HARMONIZED_QUANTIFICATION`. Choose `reflectance` only when the file contains physical BOA reflectance. These settings are alternatives, not interchangeable corrections.


In [ ]:
from examples.raster import read_pair_window, l2a_calibration, calibrate_dn, ndvi

red_path, nir_path = input_path("INPUT_B04"), input_path("INPUT_B08")
scl_path = input_path("INPUT_SCL") if os.getenv("INPUT_SCL") else None
bbox = json.loads(os.getenv("S2_AOI_BBOX", "[24.55,60.15,24.57,60.17]"))
red_dn, nir_dn, valid, grid = read_pair_window(red_path, nir_path, bbox, scl_path)
print(grid)


## 3. Convert values using the declared product convention

The metadata parser stops if it cannot establish offsets. For a legacy product without offsets, inspect the processing baseline and encode a provider-confirmed harmonized convention explicitly; do not assume all products use −1000.


In [ ]:
mode = required_env("S2_CALIBRATION")
if mode == "native_dn":
    calibration = l2a_calibration(input_path("INPUT_L2A_METADATA"))
    red = calibrate_dn(red_dn, *calibration["B04"])
    nir = calibrate_dn(nir_dn, *calibration["B08"])
elif mode == "harmonized_dn":
    scale = float(required_env("S2_HARMONIZED_QUANTIFICATION"))
    calibration = {"provider_confirmed_scale": scale, "offset": 0}
    red, nir = calibrate_dn(red_dn, scale, 0), calibrate_dn(nir_dn, scale, 0)
elif mode == "reflectance":
    calibration = {"provider_confirmed_physical_reflectance": True}
    red, nir = red_dn.astype(float), nir_dn.astype(float)
else:
    raise ValueError("S2_CALIBRATION must be native_dn, harmonized_dn, or reflectance.")
result = ndvi(red, nir, valid)
assert np.isfinite(result).any(), "No valid pixels remain; check AOI, nodata, and masks."
pd.Series(result[np.isfinite(result)]).describe()


## 4. Visualize and save the provenance

NDVI outside [−1,1] can arise from negative reflectance or data problems. Investigate it; a color scale is not a quality filter.


In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(result, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(label="NDVI")
plt.title("Calibrated Sentinel-2 NDVI — selected AOI")
plt.show()
record_provenance("copernicus-browser-ndvi.json", input_files=[red_path.name, nir_path.name], calibration_mode=mode, calibration=calibration, bbox_wgs84=bbox, grid=grid, valid_pixels=int(np.isfinite(result).sum()), units="dimensionless")


## Interpret and verify

If SCL is omitted, the example masks nodata only and is exploratory; record this limitation and add cloud/shadow/snow filtering before publication. NDVI measures spectral greenness; it does not directly measure GHG flux, biodiversity, or water-table depth.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [Sentinel-2 calibration and product metadata](https://sentiwiki.copernicus.eu/web/s2-products)
- [CDSE Copernicus Browser](https://browser.dataspace.copernicus.eu/)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
